In [0]:
%pip install databricks-vectorsearch --quiet
dbutils.library.restartPython()

In [0]:
from databricks.vector_search.client import VectorSearchClient

CATALOG     = "p2p_databricks"
VS_ENDPOINT = "p2p"

# Index names — in silver_vs schema (standard Delta tables)
PO_INDEX  = f"{CATALOG}.silver_vs.po_merged_index"
GR_INDEX  = f"{CATALOG}.silver_vs.gr_merged_index"
INV_INDEX = f"{CATALOG}.silver_vs.inv_merged_index"

# Source tables
PO_TABLE  = f"{CATALOG}.silver_vs.purchase_order"
GR_TABLE  = f"{CATALOG}.silver_vs.good_receipt"
INV_TABLE = f"{CATALOG}.silver_vs.invoice"

vsc = VectorSearchClient(disable_notice=True)

endpoint = vsc.get_endpoint(VS_ENDPOINT)
print(f"Endpoint : {endpoint['name']}")
print(f"Status   : {endpoint['endpoint_status']['state']}")

existing_indexes = [i["name"] for i in
                    vsc.list_indexes(VS_ENDPOINT).get("vector_indexes", [])]
print(f"\nExisting indexes: {existing_indexes}")
print("✅ Ready")

In [0]:
index_configs = [
    {
        "index_name":   PO_INDEX,
        "source_table": PO_TABLE,
        "primary_key":  "po_number",
        "label":        "PO"
    },
    {
        "index_name":   GR_INDEX,
        "source_table": GR_TABLE,
        "primary_key":  "gr_number",
        "label":        "GR"
    },
    {
        "index_name":   INV_INDEX,
        "source_table": INV_TABLE,
        "primary_key":  "invoice_number",
        "label":        "Invoice"
    },
]

for cfg in index_configs:
    print(f"\nCreating {cfg['label']} index...")
    try:
        if cfg["index_name"] not in existing_indexes:
            vsc.create_delta_sync_index(
                endpoint_name                 = VS_ENDPOINT,
                index_name                    = cfg["index_name"],
                source_table_name             = cfg["source_table"],
                pipeline_type                 = "TRIGGERED",
                primary_key                   = cfg["primary_key"],
                embedding_source_column       = "merged_description",
                embedding_model_endpoint_name = "databricks-bge-large-en"
            )
            print(f"  ✅ Created: {cfg['index_name']}")
        else:
            print(f"  ℹ️  Already exists — syncing")
            vsc.get_index(VS_ENDPOINT, cfg["index_name"]).sync()
    except Exception as e:
        print(f"  ❌ Error: {e}")

In [0]:
import time

print("Waiting for indexes to sync (3-5 min)...\n")

for cfg in index_configs:
    print(f"  {cfg['label']}...", end=" ", flush=True)
    for _ in range(20):
        try:
            status = (vsc.get_index(VS_ENDPOINT, cfg["index_name"])
                         .describe()
                         .get("status", {})
                         .get("ready", False))
            if status:
                print("✅ READY")
                break
            print(".", end="", flush=True)
        except Exception:
            print(".", end="", flush=True)
        time.sleep(30)
    else:
        print("⚠️  Not ready after 10 min")

In [0]:
# Test PO index — search using an invoice's merged_description
# Should return the matching PO

test_inv = (spark.table(f"{CATALOG}.silver_vs.invoice")
                 .filter("invoice_number = 'INV-2025-0001'")
                 .select("invoice_number", "po_number",
                         "vendor_name", "merged_description")
                 .first())

print(f"Test invoice    : {test_inv['invoice_number']}")
print(f"Expected PO     : {test_inv['po_number']}")
print(f"Invoice vendor  : {test_inv['vendor_name']}")
print(f"Query text      : {test_inv['merged_description'][:100]}...")
print()

po_index = vsc.get_index(VS_ENDPOINT, PO_INDEX)
results  = po_index.similarity_search(
    query_text  = test_inv["merged_description"],
    columns     = ["po_number", "vendor_name", "merged_description"],
    num_results = 3
)

print("Top 3 VS results:")
print("─" * 65)
for r in results.get("result", {}).get("data_array", []):
    match_flag = "✅" if r[0] == test_inv["po_number"] else "  "
    print(f"  {match_flag} PO: {r[0]:20s} | Vendor: {r[1]:35s} | Score: {r[3]:.4f}")

In [0]:
# Test all three indexes
CATALOG     = "p2p_databricks"
VS_ENDPOINT = "p2p"
PO_INDEX    = f"{CATALOG}.silver_vs.po_merged_index"
GR_INDEX    = f"{CATALOG}.silver_vs.gr_merged_index"
INV_INDEX   = f"{CATALOG}.silver_vs.inv_merged_index"

from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

print("=" * 65)
print("TEST 1 — PO index: query with invoice merged_description")
print("=" * 65)

test_inv = (spark.table(f"{CATALOG}.silver_vs.invoice")
                 .filter("invoice_number = 'INV-2025-0001'")
                 .select("invoice_number", "po_number",
                         "vendor_name", "merged_description")
                 .first())

po_index = vsc.get_index(VS_ENDPOINT, PO_INDEX)
results  = po_index.similarity_search(
    query_text  = test_inv["merged_description"],
    columns     = ["po_number", "vendor_name"],
    num_results = 3
)

print(f"Invoice       : {test_inv['invoice_number']}")
print(f"Expected PO   : {test_inv['po_number']}")
print(f"Invoice vendor: {test_inv['vendor_name']}")
print()
for r in results.get("result", {}).get("data_array", []):
    match = "✅ CORRECT" if r[0] == test_inv["po_number"] else "  "
    print(f"  {match} PO: {r[0]:20s} | Vendor: {r[1]:30s} | Score: {r[2]:.4f}")

print()
print("=" * 65)
print("TEST 2 — Wrong vendor scenario: does VS catch it?")
print("=" * 65)

# Pick a wrong-vendor invoice — its vendor should NOT match the PO
wrong_vendor_inv = (spark.table(f"{CATALOG}.silver_vs.invoice")
                        .filter("invoice_number = 'INV-2025-0049'")
                        .select("invoice_number", "po_number",
                                "vendor_name", "merged_description")
                        .first())

results2 = po_index.similarity_search(
    query_text  = wrong_vendor_inv["merged_description"],
    columns     = ["po_number", "vendor_name"],
    num_results = 3
)

print(f"Invoice       : {wrong_vendor_inv['invoice_number']}")
print(f"PO reference  : {wrong_vendor_inv['po_number']}")
print(f"Invoice vendor: {wrong_vendor_inv['vendor_name']}")
print()
for r in results2.get("result", {}).get("data_array", []):
    match = "✅ PO match" if r[0] == wrong_vendor_inv["po_number"] else "  "
    print(f"  {match} PO: {r[0]:20s} | Vendor: {r[1]:30s} | Score: {r[2]:.4f}")

print()
print("=" * 65)
print("TEST 3 — Duplicate invoice: does VS find the original?")
print("=" * 65)

dup_inv = (spark.table(f"{CATALOG}.silver_vs.invoice")
               .filter("invoice_number = 'INV-2025-0053-DUP'")
               .select("invoice_number", "po_number",
                       "vendor_name", "merged_description")
               .first())

inv_index = vsc.get_index(VS_ENDPOINT, INV_INDEX)
results3  = inv_index.similarity_search(
    query_text  = dup_inv["merged_description"],
    columns     = ["invoice_number", "po_number", "vendor_name"],
    num_results = 3
)

print(f"DUP invoice   : {dup_inv['invoice_number']}")
print(f"PO reference  : {dup_inv['po_number']}")
print()
for r in results3.get("result", {}).get("data_array", []):
    is_orig = r[0] == dup_inv["invoice_number"].replace("-DUP", "")
    flag    = "✅ ORIGINAL" if is_orig else "  "
    print(f"  {match} PO: {r[0]:20s} | Vendor: {r[1]:30s} | Score: {r[2]}")